# **SpaceX Falcon 9 First Stage Landing Prediction: Interactive Dashboard**

In this notebook, I'll create an interactive dashboard using Dash, a Python framework built on top of Flask, Plotly.js, and React.js for building web applications. This dashboard will visualize SpaceX launch data and allow users to explore the relationships between launch sites, payload masses, and success rates interactively.

## Setting Up the Environment

First, I'll mount my Google Drive to access the necessary data files. This step is only needed when working in Google Colab.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Next, I'll navigate to the project directory where I've stored my data files. This ensures that all file paths will be relative to this project folder.

In [2]:
%cd "/content/drive/Othercomputers/My MacBook Pro/portfolio/spacex"
file_prefix = %pwd

/content/drive/Othercomputers/My MacBook Pro/portfolio/spacex


## Installing Required Packages

I'll need to install the Dash package, which is the framework I'll use to create the interactive dashboard. If you're running this notebook locally and already have Dash installed, you can skip this step.

In [5]:
!pip install dash
!pip install jupyter-dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.0 MB/s eta 0:00:00


## Importing Libraries

Now I'll import all the necessary libraries for our dashboard:
- `jupyter_dash`: Allows running Dash applications within Jupyter notebooks
- `dash`, `dcc`, `html`: Core Dash components for creating web applications
- `Input`, `Output`: For creating interactive callbacks
- `pandas`: For data manipulation
- `plotly.express`: For creating interactive visualizations

In [6]:
import jupyter_dash
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

## Loading and Preparing the Data

I'll now load the SpaceX launch data from the CSV file that I prepared in the previous notebooks. Additionally, I'll calculate the maximum and minimum payload masses, which will be used to set the range for our payload slider.

In [7]:
# read the launch data into pandas dataframe
# identify maximum and minimum payloads
spacex_df = pd.read_csv("data/spacex_launch_dash.csv")
max_payload = spacex_df["Payload Mass (kg)"].max()
min_payload = spacex_df["Payload Mass (kg)"].min()

## Initializing the Dash Application

Here I'll create a new Dash application instance. This is the core object that will be used to build our dashboard.

In [8]:
# create the dash application
app = Dash(__name__)

## Setting Up Launch Site Options

Before creating the layout, I'll define the options for our dropdown menu. This will allow users to select which launch site's data they want to view. I include an "All Sites" option to allow viewing aggregated data across all sites.

In [9]:
# list of dictionaries of launch site options
launch_site_options = [{"label": "All Sites",     "value": "ALL"},
                       {"label": "CCAFS LC-40",   "value": "CCAFS LC-40"},
                       {"label": "VAFB SLC-4E",   "value": "VAFB SLC-4E"},
                       {"label": "KSC LC-39A",    "value": "KSC LC-39A"},
                       {"label": "CCAFS SLC-40",  "value": "CCAFS SLC-40"}]

## Creating the Dashboard Layout

Now I'll design the layout of our dashboard using HTML and Dash components. The layout includes:
1. A title header
2. A dropdown menu to select launch sites
3. A pie chart showing success rates
4. A slider to filter by payload mass range
5. A scatter plot showing the relationship between payload mass and success

The design uses a hierarchical structure of HTML Div elements to organize the various components.

In [10]:
# create app layout
app.layout = html.Div(children = [html.H1(
                                      "SpaceX Launch Records Dashboard",
                                      style = {"testAlign": "center",
                                                "color": "#4f86f7",
                                                "font-size": 40}
                                  ),
                                  html.Div(
                                      dcc.Dropdown(
                                          id = "site-dropdown",
                                          options = launch_site_options,
                                          value = "ALL",
                                          placeholder = "Select launch site",
                                          searchable = True)
                                  ),
                                  html.Br(),
                                  html.Div(
                                      dcc.Graph(
                                          id = "success-pie-chart")
                                  ),
                                  html.Br(),
                                  html.P(
                                      "Payload range (kg)",
                                      style = {"color": "#4f86f7",
                                               "font-size": 20}
                                  ),
                                  html.Div(
                                      dcc.RangeSlider(
                                          id = "payload-slider",
                                          min = 0,
                                          max = 10000,
                                          step = 1000,
                                          marks = {0:     "0",
                                                   1000:  "1000",
                                                   2000:  "2000",
                                                   3000:  "3000",
                                                   4000:  "4000",
                                                   5000:  "5000",
                                                   6000:  "6000",
                                                   7000:  "7000",
                                                   8000:  "8000",
                                                   9000:  "9000",
                                                   10000: "10000"},
                                          value = [min_payload, max_payload])
                                  ),
                                  html.Div(
                                      dcc.Graph(
                                          id = "success-payload-scatter-chart")
                                  )])

## Implementing Interactive Callbacks

Now I'll introduce interactivity into the dashboard using Dash callbacks. These functions will update the visualizations based on user input.

### Callback 1: Pie Chart by Launch Site

This callback creates a pie chart showing success rates, which updates when the user selects a different launch site from the dropdown. The chart displays:
- When "All Sites" is selected: A pie chart showing the number of successful launches by site
- When a specific site is selected: A pie chart showing the proportion of successful vs. failed launches at that site

In [11]:
# define function to plot pie chart of successful launches for all sites or
# specified particular site
# use callback decorator to update plot when site is updated
@app.callback(
    Output(component_id = "success-pie-chart",
           component_property = "figure"),
    Input(component_id = "site-dropdown",
          component_property = "value")
)
def get_pie_chart(entered_site):
  if entered_site == "ALL":
    filtered_df = spacex_df.groupby("Launch Site")["class"].sum().reset_index()
    fig = px.pie(
        filtered_df,
        values = "class",
        names = "Launch Site",
        title = "Successful Launches by Launch Site")
  else:
    filtered_df = spacex_df[spacex_df["Launch Site"] == entered_site]
    number_of_launches = len(filtered_df)
    number_of_successes = filtered_df["class"].sum()
    number_of_failures = number_of_launches - number_of_successes
    df_pie = pd.DataFrame(
        {"count": [number_of_successes, number_of_failures]},
        index = ["successes", "failures"])
    fig = px.pie(
        df_pie,
        values = "count",
        names = df_pie.index,
        title = "Proportion of Failed and Successful Launches at Launch Site {}".format(entered_site))
  return fig

### Callback 2: Scatter Plot of Payload vs. Success

This callback creates a scatter plot showing the relationship between payload mass and mission success, which updates based on both:
1. The selected launch site from the dropdown
2. The payload range selected from the slider

The x-axis shows payload mass, the y-axis shows success (0 or 1), and points are colored by booster version. This visualization helps identify whether certain payload ranges have higher success rates and whether this varies by launch site or booster version.

In [12]:
# function to plot scatter plot of success versus payload mass either at all
# sites or at user-specified site
@app.callback(
    Output(component_id = "success-payload-scatter-chart",
           component_property = "figure"),
    [Input(component_id = "site-dropdown",
           component_property = "value"),
     Input(component_id = "payload-slider",
           component_property = "value")]
)
def get_scatter_plot(entered_site, payload_range):
  if entered_site == "ALL":
    filtered_df = spacex_df[["Payload Mass (kg)",
                            "class",
                            "Booster Version Category"]]
    fig = px.scatter(
        filtered_df,
        x = "Payload Mass (kg)",
        y = "class",
        color = "Booster Version Category",
        title = "Correlation Between Payload and Success for All Sites")
  else:
    filtered_df = spacex_df[spacex_df["Launch Site"] == entered_site]
    filtered_df = filtered_df[["Payload Mass (kg)",
                               "class",
                               "Booster Version Category"]]
    fig = px.scatter(
        filtered_df,
        x = "Payload Mass (kg)",
        y = "class",
        color = "Booster Version Category",
        title = "Correlation Between Payload and Success at Launch Site {}".format(entered_site))
  fig.update_xaxes(range = payload_range)
  return fig

## Running the Dashboard

Finally, I'll run the Dash application. This will start a local web server and display the interactive dashboard. In a notebook environment, the dashboard will be embedded directly in the notebook output cell. If running this in a standalone Python script, it would open in a web browser.

The `debug=True` parameter enables hot-reloading during development, while `use_reloader=False` prevents the application from restarting when running in a Jupyter notebook environment.

In [14]:
# run the app
if __name__ == "__main__":
  app.run(debug = True,
                 use_reloader = False)

<IPython.core.display.Javascript object>

## Dashboard Usage and Analysis

Once the dashboard is running, you can interact with it to explore the SpaceX launch data:

1. Use the dropdown menu to switch between viewing all launch sites or focusing on a specific site
2. Observe how the pie chart changes to show either:
   - The distribution of successful launches across sites when "All Sites" is selected
   - The success vs. failure ratio at a specific site when that site is selected
3. Use the payload range slider to filter the scatter plot by payload mass
4. Examine the scatter plot to identify any patterns between payload mass and mission success
5. Note how different booster versions (indicated by color) perform with different payload masses

This interactive visualization provides valuable insights into factors affecting SpaceX launch success rates, which will inform our predictive modeling in subsequent notebooks.